# Bài 18 · Vision Transformer: cắt ảnh thành mảnh, đọc bằng Transformer, rồi so với CNN

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/18-vision-transformer.ipynb)

Notebook đi kèm [Bài 18 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/vision-transformer/) (demo tương tác nằm ở trang bài học).
ViT (Dosovitskiy và cộng sự, 2020) đọc một bức ảnh như đọc một câu: cắt ảnh thành các mảnh vuông, mỗi mảnh thành
một token, rồi cho cả dãy đi qua các khối Transformer của Bài 17. Ta tự viết ViT bằng numpy, chạy hai ViT đã huấn
luyện trên MNIST thật, rồi hỏi: không có tích chập thì mô hình biết mảnh nào nằm ở đâu bằng cách nào, và cái giá
của việc không cài sẵn hiểu biết về không gian là gì?

**Bạn sẽ làm:**
1. Cắt một chữ số thành lưới mảnh, đếm token, tham số và phép nhân–cộng, và kiểm tra rằng “duỗi mảnh rồi nhân với
   $E$” đúng bằng một lớp tích chập.
2. Tự viết lượt truyền xuôi của ViT (token [CLS], vectơ vị trí, khối Transformer không mặt nạ) và chạy hai ViT
   đã huấn luyện của demo trên 10.000 ảnh test.
3. Xáo thứ tự các mảnh, rồi xáo điểm ảnh trong mỗi mảnh, để thấy vectơ vị trí làm gì.
4. Đọc vectơ vị trí và attention của ViT MNIST (tự tính) và của một ViT‑Ti/16 thật (số đo có sẵn).
5. Tự huấn luyện một ViT nhỏ, rồi so ViT với LeNet và MLP khi số ảnh huấn luyện tăng từ 1.000 lên 55.000.

**Cần biết trước:** [Bài 17 · Transformer & GPT](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/transformer/)
(tự chú ý nhiều đầu, LayerNorm, khối Transformer), [Bài 11 · Mạng tích chập](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/cnn/)
và [Bài 12 · Pooling & kiến trúc CNN kinh điển](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/kien-truc-cnn/)
(LeNet, mô hình để so sánh). **Thời gian chạy:** dưới 1 phút trên CPU, kể cả tải MNIST (~11 MB) lần đầu.

Mọi con số trong notebook đều do chính các ô bên dưới tính ra (hoặc đọc từ đúng các tệp mà demo dùng), và trùng với
số trong demo. Việc huấn luyện ViT, LeNet và MLP ở 6 cỡ dữ liệu × 3 seed mất hơn một giờ trên máy nhiều lõi, nên notebook
nạp trọng số của hai ViT đã huấn luyện và bảng kết quả. ViT‑Ti/16 thật không chạy ở đây: trọng số của nó không được
phát hành kèm, chỉ các số đo đã tính sẵn từ nó.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

Chỉ cần numpy và matplotlib (Colab đã cài sẵn). Hàm `vn()` in số theo kiểu Việt Nam: dấu phẩy thập phân,
dấu chấm hàng nghìn. Python vẫn *gõ* số thập phân bằng dấu chấm (`0.1`), chỉ phần in ra là đổi.

In [ ]:
import base64
import gzip
import hashlib
import io
import json
import math
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter

# Bảng màu của khóa học (an toàn cho người mù màu); luôn đi kèm một mã hóa thứ hai (hình dạng, nét gạch, nhãn)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
DIVERGING = LinearSegmentedColormap.from_list("cam_xam_xanh", [ORANGE, "#f2f2f2", BLUE])  # âm → 0 → dương
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38 (không in “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return s.replace("-", "−")


def pct(p, digits=2):
    return vn(100 * p, digits) + "%"


def vn_tick(v, _pos=None):
    """Nhãn vạch trên trục: 0,5 · −2 · 60.000 (bỏ số 0 thừa ở cuối)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return vn(v, len(s.split(".")[1]) if "." in s else 0)


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def log_tick(v, _pos=None):
    """Nhãn vạch trục log: 0,5 · 20 · 10⁻¹²."""
    if 0.01 <= v <= 1000:
        return vn_tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{vn_tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def sci(x, digits=1):
    """Sai số rất nhỏ kiểu Việt Nam: 3,3·10⁻¹⁶ (số từ 0,001 trở lên in thường)."""
    if x == 0:
        return "0"
    if abs(x) >= 1e-3:
        return vn(x, 3 if abs(x) < 1 else 2)
    m, e = f"{x:.{digits}e}".split("e")
    return f"{vn(float(m), digits)}·10{str(int(e)).translate(SUP)}"


def vn_axes(fig):
    """Đổi nhãn mọi trục của hình (cả thanh màu) sang kiểu Việt Nam; gọi ngay trước plt.show().
    Trục log lấy vạch 1–2–5 nếu ngắn, hoặc các lũy thừa của 10 (cách đều, tối đa ~9 vạch) nếu dài."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(vn_tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
                if step > 1:
                    axis.set_minor_locator(NullLocator())
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


print(vn(13007, 0), vn(0.1, 1), vn(-4.3805, 4), pct(0.993527), sci(3.3e-16))

## 2 · Dữ liệu

MNIST (LeCun và cộng sự, 1998): 60.000 ảnh huấn luyện và 10.000 ảnh test, chữ số viết tay xám 28×28. Ô dưới tải
4 tệp gốc và kiểm tra mã MD5, nên ai chạy cũng có đúng cùng một dữ liệu. Ô cũng nạp ba tệp mà demo dùng
(`fetch_json`, có kiểm tra MD5):
- `vit-mnist-weights.json`: hai ViT đã huấn luyện trên 55.000 ảnh (seed 0), một có vectơ vị trí và một không,
  trọng số làm tròn về float16, kèm 64 ảnh test đầu tiên;
- `vit-mnist.json`: kết quả của ViT, LeNet và MLP với 1.000–55.000 ảnh huấn luyện × 3 seed, và các thí nghiệm phụ;
- `vit-pretrained.json`: số đo bên trong ViT‑Ti/16 thật trên 8 ảnh chụp thuộc phạm vi công cộng.

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # mã MD5 chuẩn của MNIST
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"


def load(kind):
    """Định dạng IDX: 16 byte đầu của tệp ảnh (8 byte của tệp nhãn) là phần mô tả, còn lại là các byte 0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


# Kết quả của những lần huấn luyện dài (cùng tệp với demo trên web). build.py thay COURSE_DATA_URL bằng thư mục data/
# của repo notebook công khai.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"
    return json.loads(p.read_text())


fetch()
shipped = fetch_json("vit-mnist-weights.json", "74ab5fb70eee71f88aee5b93b4d9ccd7")  # 2 ViT đã huấn luyện + 64 ảnh
runs = fetch_json("vit-mnist.json", "00448109c07dc2e8aee1caad93f12b68")  # ViT / LeNet / MLP × cỡ dữ liệu × seed
pre = fetch_json("vit-pretrained.json", "003ceb0d264e7b2f21195b864801b8c3")  # số đo của ViT-Ti/16 thật

Trọng số được lưu dạng float16 mã hóa base64. Ta giải mã thành hai bản: float64 để so từng chữ số thập phân với
script đã huấn luyện, và float32 (nhanh gấp đôi) để chạy trên cả tập test.

In [ ]:
train_img, train_lab = load("train")
test_img, test_lab = load("t10k")
Xte = (test_img / 255.0).astype(np.float32)  # (10.000, 28, 28), giá trị trong [0, 1]
yte = test_lab.astype(np.int64)


def unpack(d, dtype):
    return np.frombuffer(base64.b64decode(d["b64"]), "<f2").reshape(d["shape"]).astype(dtype)


cfg = shipped["cfg"]
P, D, L, HEADS, MLP = cfg["P"], cfg["D"], cfg["L"], cfg["heads"], cfg["mlp"]
vit64 = {name: {k: unpack(v, np.float64) for k, v in m.items()} for name, m in shipped["models"].items()}
vit32 = {name: {k: unpack(v, np.float32) for k, v in m.items()} for name, m in shipped["models"].items()}
digits = np.frombuffer(base64.b64decode(shipped["digits"]["x"]), np.uint8).reshape(-1, 28, 28)
y64 = np.array(shipped["digits"]["y"])
assert np.array_equal(digits, test_img[:64]) and np.array_equal(y64, yte[:64])  # đúng 64 ảnh test đầu tiên

print(f"huấn luyện: {vn(len(train_img), 0)} ảnh, test: {vn(len(test_img), 0)} ảnh, mỗi ảnh {train_img.shape[1:]}")
print(f"ViT đã huấn luyện: mảnh {P}×{P}, D = {D}, {L} khối × {HEADS} đầu, MLP {MLP}; hai bản: {list(vit64)}")
n_runs = sum(len(g["sweep"]) for g in runs["groups"]) + 2 * len(runs["groups"])
print(f"tệp kết quả: {len(runs['groups'])} cấu hình × 3 seed, tổng cộng {n_runs} lần huấn luyện (kể cả dò η)")
su = runs["setup"]
print(
    f"cỡ tập huấn luyện: {' · '.join(vn(n, 0) for n in su['sizes'])} ảnh; mỗi lần {vn(su['steps'], 0)} bước AdamW, lô {su['batch']}"
)
print(
    f"suy giảm trọng số {vn(su['wd'], 2)}; tăng dần η trong {pct(su['warm'] / su['steps'], 0)} số bước đầu rồi giảm theo cosin"
)

16 trong 64 ảnh test mà demo dùng, mỗi ảnh ghi nhãn đúng ở trên:

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(9, 2.6))
for ax, img, lab in zip(axes.flat, digits, y64):
    ax.imshow(img, cmap="gray_r")
    ax.set_title(str(lab), fontsize=11)
    ax.axis("off")
plt.show()

## 3 · Cắt ảnh thành lưới mảnh

Transformer ở Bài 17 đọc một dãy token. Muốn đưa ảnh vào, ViT cắt ảnh $H \times W$ thành các mảnh vuông
$P \times P$ không chồng nhau, rồi xếp chúng theo thứ tự đọc (trái sang phải, trên xuống dưới):

$$N = \frac{HW}{P^2}$$

Với chữ số 28×28 và $P = 4$, ta được lưới 7×7 = 49 mảnh, mỗi mảnh duỗi thành một vectơ hàng 16 số. Ô dưới viết
việc cắt bằng vòng lặp cho dễ đọc, sau đó viết lại bằng numpy: bản numpy chỉ đổi hình dạng mảng, rồi ta kiểm tra
hai cách cho cùng kết quả.

In [ ]:
def patchify_loop(x, P):
    """Một ảnh (28, 28) → (N, P²): duyệt lưới mảnh theo hàng, duỗi mỗi mảnh theo hàng."""
    G = x.shape[0] // P
    rows = []
    for gi in range(G):  # hàng của lưới mảnh
        for gj in range(G):  # cột của lưới mảnh
            rows.append(x[gi * P : (gi + 1) * P, gj * P : (gj + 1) * P].reshape(-1))
    return np.array(rows)


def patchify(X, P):
    """Nhiều ảnh (B, 28, 28) → (B, N, P²), đúng thứ tự của patchify_loop."""
    B, H, _ = X.shape
    G = H // P
    return X.reshape(B, G, P, G, P).transpose(0, 1, 3, 2, 4).reshape(B, G * G, P * P)


x0 = Xte[0]
assert np.array_equal(patchify_loop(x0, 4), patchify(Xte[:1], 4)[0])
for size in (4, 7, 14):
    pts = patchify(Xte[:1], size)[0]
    print(
        f"mảnh {size:>2}×{size:<2}: lưới {28 // size}×{28 // size} = {len(pts):>2} mảnh, cộng [CLS] thành "
        f"{len(pts) + 1:>2} token; mỗi mảnh {size * size:>3} số; {int((pts.max(1) == 0).sum()):>2} mảnh chỉ có nền"
    )

Hình dưới đặt ảnh (trái, đường xanh là lưới mảnh 4×4) cạnh dãy token mà Transformer thực sự nhận (phải): ô xanh
đầu tiên là token [CLS] (mục 5), sau đó là 49 mảnh theo thứ tự đọc. Mảnh chỉ có nền là ô trắng viền xám.
Chúng vẫn là token như mọi mảnh khác, vẫn chiếm một hàng và một cột trong ma trận chú ý.

In [ ]:
pts = patchify(Xte[:1], P)[0]
fig = plt.figure(figsize=(10, 2.9))
ax = fig.add_axes([0.0, 0.04, 0.24, 0.84])
ax.imshow(x0, cmap="gray_r", vmin=0, vmax=1)
for k in range(1, 28 // P):
    ax.axhline(k * P - 0.5, color=BLUE, lw=0.9)
    ax.axvline(k * P - 0.5, color=BLUE, lw=0.9)
ax.set_title(f"chữ số {yte[0]}, lưới 7×7 mảnh 4×4", fontsize=10)
ax.axis("off")
per_row, step = 25, P + 1  # dãy 50 token xếp thành 2 hàng × 25 ô, cách nhau 1 điểm ảnh
mosaic = np.full((2 * step, per_row * step), np.nan)
for t in range(1, 50):
    r_, c_ = divmod(t, per_row)
    mosaic[r_ * step : r_ * step + P, c_ * step : c_ * step + P] = pts[t - 1].reshape(P, P)
cmap = plt.get_cmap("gray_r").copy()
cmap.set_bad("white")
ax = fig.add_axes([0.28, 0.04, 0.72, 0.84])
ax.imshow(mosaic, cmap=cmap, vmin=0, vmax=1)
for t in range(50):
    r_, c_ = divmod(t, per_row)
    ax.add_patch(
        Rectangle((c_ * step - 0.5, r_ * step - 0.5), P, P, fc=BLUE if t == 0 else "none", ec=GREY, lw=0.7, alpha=0.9)
    )
ax.text(1.5, 1.5, "CLS", color="white", fontsize=6, ha="center", va="center", fontweight="bold")
ax.set_title("dãy 50 token: [CLS], rồi 49 mảnh theo thứ tự đọc", fontsize=10)
ax.axis("off")
plt.show()

Ảnh màu thật cũng được cắt y như vậy. ViT‑B/16 nhận ảnh 224×224 với $C = 3$ kênh màu. Mỗi mảnh 16×16 là
$P^2 C$ số, và số token tăng theo bình phương khi mảnh nhỏ lại:

In [ ]:
def n_tokens(img, P):
    return (img // P) ** 2 + 1  # N mảnh + [CLS]


print(f"ViT-B/16 ở 224×224: mỗi mảnh 16·16·3 = {16 * 16 * 3} số; {n_tokens(224, 16)} token")
print(f"tinh chỉnh ở 384×384 (như bài báo): {n_tokens(384, 16)} token")
ratio = (n_tokens(224, 8) / n_tokens(224, 16)) ** 2
print(f"mảnh 8×8 thay vì 16×16: {n_tokens(224, 8)} token, ma trận chú ý T×T lớn gấp {vn(ratio, 1)} lần")
print(f"mỗi điểm ảnh một token: {vn(n_tokens(224, 1), 0)} token")
assert (16 * 16 * 3, n_tokens(224, 16), n_tokens(384, 16), n_tokens(224, 8)) == (768, 197, 577, 785)

Thu cạnh mảnh còn một nửa thì số token gấp 4, nên ma trận chú ý (mỗi token một hàng, một cột) gấp gần 16.
Đây là lý do ViT dùng mảnh to: mỗi điểm ảnh một token thì ảnh 224×224 thành 50.177 token.

## 4 · Mỗi mảnh thành một token, và đó thực chất là một phép tích chập

Mỗi mảnh $x_p^i$ (vectơ hàng $P^2 C$ số) nhân với cùng một ma trận học được $E \in \mathbb{R}^{(P^2 C) \times D}$
để thành token $x_p^i E$ có $D$ số (ViT trên MNIST của ta: $D = 48$; ViT‑B: 768). Cột $d$ của $E$ là $P^2 C$ trọng
số; xếp lại thành khối $P \times P \times C$ thì nó là một **bộ lọc**. Như vậy “duỗi mảnh rồi nhân với $E$” chính là
một lớp tích chập (Bài 11) có $D$ bộ lọc $P \times P$, bước trượt $P$, không đệm: các cửa sổ không chồng lên nhau.
Mã nguồn gốc của ViT viết đúng như vậy (`nn.Conv(..., kernel_size=patch, strides=patch, padding="VALID")`).

Ô dưới lấy $E$ của mô hình đã huấn luyện, tính tích chập bằng vòng lặp trên từng ô đầu ra, rồi so với phép nhân ma
trận.

In [ ]:
E = vit64["pos"]["We"]  # (16, 48): 16 điểm ảnh của một mảnh → 48 số của token
x = digits[0] / 255.0  # float64, như script của demo
G = 28 // P
conv = np.zeros((G, G, D))
for i in range(G):  # cửa sổ thứ (i, j): bước trượt P nên các cửa sổ không chồng nhau
    for j in range(G):
        window = x[i * P : (i + 1) * P, j * P : (j + 1) * P]
        for d in range(D):  # bộ lọc d = cột d của E, xếp lại thành 4×4
            conv[i, j, d] = (window * E[:, d].reshape(P, P)).sum()
matmul = patchify(x[None], P)[0] @ E  # (49, 48)
err = np.abs(conv.reshape(G * G, D) - matmul).max()
print(f"tích chập {D} bộ lọc {P}×{P}, bước {P}, không đệm: đầu ra {conv.shape}, duỗi thành {conv.reshape(-1, D).shape}")
print(f"duỗi mảnh rồi nhân với E: {matmul.shape}; chênh lệch lớn nhất {sci(err)}")
assert err < 1e-12

Hai cách cho cùng một kết quả, vì chúng làm đúng cùng các phép nhân–cộng, chỉ khác cách đánh chỉ số. Ảnh màu chỉ thêm
một trục kênh: mảnh duỗi thành $P^2 C$ số theo đúng thứ tự (kênh, hàng, cột) mà bộ lọc được duỗi. Ô dưới kiểm tra
điều đó trên một ảnh màu ngẫu nhiên 64×64 với 8 bộ lọc 16×16×3, rồi in chênh lệch mà script của demo đo trên ViT‑Ti/16
thật (192 bộ lọc 16×16×3, trên 8 ảnh chụp).

In [ ]:
def patch_rows_rgb(img, P):
    """Ảnh màu (C, H, W) → (N, C·P·P): mỗi mảnh duỗi theo thứ tự (kênh, hàng, cột), như trọng số của lớp tích chập."""
    C, H, _ = img.shape
    G = H // P
    return img.reshape(C, G, P, G, P).transpose(1, 3, 0, 2, 4).reshape(G * G, C * P * P)


rng = np.random.default_rng(0)
img = rng.random((3, 64, 64))
filters = rng.normal(0, 0.02, (8, 3, 16, 16))  # 8 bộ lọc 16×16×3
conv3 = np.array(
    [
        [(filters * img[:, i * 16 : (i + 1) * 16, j * 16 : (j + 1) * 16]).sum((1, 2, 3)) for j in range(4)]
        for i in range(4)
    ]
)
E3 = filters.reshape(8, -1).T  # (768, 8): cột d là bộ lọc d duỗi thẳng
err3 = np.abs(conv3.reshape(16, 8) - patch_rows_rgb(img, 16) @ E3).max()
print(f"ảnh màu: mỗi mảnh {E3.shape[0]} số; chênh lệch lớn nhất {sci(err3)}")
print(f"ViT-Ti/16 thật (đo sẵn bằng script của demo): chênh lệch lớn nhất {sci(pre['model']['convErr'])}")
assert err3 < 1e-12

Chênh lệch từ $10^{-16}$ đến $10^{-14}$ chỉ là sai số làm tròn khi cộng theo thứ tự khác. Vậy lớp đầu tiên của ViT *là* một lớp
tích chập, chỉ có điều nó là lớp tích chập duy nhất: bước trượt bằng cỡ bộ lọc, nên mỗi điểm ảnh chỉ thuộc một
token, và sau đó không còn tích chập nào nữa.

## 5 · Token [CLS], vectơ vị trí và các khối Transformer

Dãy đi vào khối đầu tiên là phương trình (1) của bài báo, viết theo quy ước hàng:

$$z_0 = \big[\,x_{\text{class}};\; x_p^{1}E;\; \dots;\; x_p^{N}E\,\big] + E_{\text{pos}}$$

- $x_{\text{class}}$ là token **[CLS]**: một vectơ $D$ số học được, đặt ở đầu dãy, giống token [CLS] của BERT.
  Nó không thuộc mảnh nào; qua các khối, nó gom thông tin từ mọi mảnh, và chỉ đầu ra của nó ở lớp cuối đi vào bộ
  phân loại.
- $E_{\text{pos}}$ gồm $N + 1$ **vectơ vị trí** học được, mỗi vị trí một vectơ, cộng vào từng hàng. Lúc đầu
  chúng là số ngẫu nhiên nhỏ, không chứa thông tin nào về lưới 2‑D.

Rồi $L$ khối Transformer như Bài 17, kiểu pre‑LN: $h \leftarrow h + \mathrm{MSA}(\mathrm{LN}(h))$, rồi
$h \leftarrow h + \mathrm{MLP}(\mathrm{LN}(h))$. Khác GPT ở một điểm: **không có mặt nạ**. Token nào cũng nhìn được
mọi token, cả những mảnh đứng sau nó.

Ô dưới viết lượt truyền xuôi đúng như script đã huấn luyện mô hình (`reel/vit/vit.py`): LayerNorm với
$\epsilon = 10^{-6}$, GELU xấp xỉ bằng tanh. Hàm có thêm tùy chọn trả về attention (mục 8) và bộ nhớ đệm cho lan
truyền ngược (mục 10).

In [ ]:
C_GELU = float(np.sqrt(2 / np.pi))


def layer_norm(h, g, b, eps=1e-6):
    hc = h - h.mean(-1, keepdims=True)
    inv = 1 / np.sqrt((hc * hc).mean(-1, keepdims=True) + eps)
    return hc * inv * g + b, (hc * inv, inv)


def gelu(u):  # xấp xỉ bằng tanh, như GPT-2
    t = np.tanh(C_GELU * (u + 0.044715 * u * u * u))
    return 0.5 * u * (1 + t), t


def forward(p, X, P=P, heads=HEADS, maps=False, cache=False):
    """ViT trên ảnh X (B, 28, 28) → logit (B, 10). maps=True trả thêm attention (B, đầu, T, T) của từng khối."""
    xp = patchify(X, P)
    B, N, _ = xp.shape
    Dm = p["We"].shape[1]
    dh = Dm // heads
    h = xp @ p["We"] + p["be"]  # mỗi mảnh thành một token: x_p E
    h = np.concatenate([np.broadcast_to(p["cls"], (B, 1, Dm)), h], 1)  # token [CLS] đứng đầu
    if "pos" in p:
        h = h + p["pos"]  # + E_pos
    T, saved, att = N + 1, [], []
    for l in range(sum(k.endswith(".Wqkv") for k in p)):
        w = {k.split(".", 1)[1]: v for k, v in p.items() if k.startswith(f"{l}.")}
        a, c1 = layer_norm(h, w["g1"], w["b1"])
        q, k, v = (a @ w["Wqkv"] + w["bqkv"]).reshape(B, T, 3, heads, dh).transpose(2, 0, 3, 1, 4)
        A = q @ k.transpose(0, 1, 3, 2) * (1 / dh**0.5)  # không mặt nạ: mọi token nhìn mọi token
        A = np.exp(A - A.max(-1, keepdims=True))
        A /= A.sum(-1, keepdims=True)
        o = (A @ v).transpose(0, 2, 1, 3).reshape(B, T, Dm)
        h = h + o @ w["Wo"] + w["bo"]  # h ← h + MSA(LN h)
        m, c2 = layer_norm(h, w["g2"], w["b2"])
        u = m @ w["W1"] + w["c1"]
        gu, t = gelu(u)
        h = h + gu @ w["W2"] + w["c2"]  # h ← h + MLP(LN h)
        if maps:
            att.append(A)
        if cache:
            saved.append((a, c1, q, k, v, A, o, m, c2, u, gu, t))
    hf, cf = layer_norm(h[:, 0], p["gf"], p["bf"])  # chỉ đọc token [CLS] ở lớp cuối
    z = hf @ p["Wh"] + p["bh"]
    if cache:
        return z, (xp, saved, hf, cf)
    return (z, att) if maps else z


def predict(p, X, bs=500, **kw):
    return np.concatenate([forward(p, X[i : i + bs], **kw).argmax(1) for i in range(0, len(X), bs)])

## 6 · Đếm tham số và phép nhân–cộng

Trước khi chạy, ta đếm. Một khối Transformer có bốn ma trận $D \times D$ (Q, K, V và phép chiếu đầu ra, ở đây Q, K, V
gộp thành một ma trận $D \times 3D$), MLP $D \to M \to D$, hai LayerNorm (mỗi cái một vectơ nhân và một vectơ cộng),
và các độ lệch: $4D^2 + 2DM + 9D + M$ tham số. Ngoài các khối còn có $E$, [CLS], các vectơ vị trí, LayerNorm cuối
và bộ phân loại.

Phép nhân–cộng (MAC) của một ảnh chia làm hai phần. Phần chiếu (tạo token, Q/K/V, đầu ra, MLP) tăng theo số token
$T$. Phần chú ý (điểm $QK^\top$ và tổng có trọng số $AV$, mỗi cái $T^2 D$ MAC một khối) tăng theo $T^2$. Ta không đếm
độ lệch, LayerNorm, softmax và GELU, như ở các bài trước.

In [ ]:
def block_params(Dm, M):
    return 4 * Dm * Dm + 2 * Dm * M + 9 * Dm + M


def vit_params(P, Dm, L, M, img=28, C=1, classes=10, pos=True):
    N = (img // P) ** 2
    parts = {
        "E (chiếu mảnh)": P * P * C * Dm + Dm,
        "[CLS]": Dm,
        "vectơ vị trí": (N + 1) * Dm if pos else 0,
        f"{L} khối Transformer": L * block_params(Dm, M),
        "LN cuối + phân loại": 2 * Dm + Dm * classes + classes,
    }
    return parts, sum(parts.values())


def vit_macs(P, Dm, L, M, img=28, C=1, classes=10):
    N = (img // P) ** 2
    T = N + 1
    attention = L * 2 * T * T * Dm
    rest = N * P * P * C * Dm + L * (4 * T * Dm * Dm + 2 * T * Dm * M) + Dm * classes
    return attention, rest


parts, n_pos = vit_params(P, D, L, MLP)
for k, v in parts.items():
    print(f"{k:<22} {vn(v, 0):>7}")
n_shipped = {name: sum(a.size for a in p.values()) for name, p in vit64.items()}
_, n_nopos = vit_params(P, D, L, MLP, pos=False)
print(f"tổng: {vn(n_pos, 0)} tham số (đếm trong tệp: {vn(n_shipped['pos'], 0)}); không vị trí: {vn(n_nopos, 0)}")
assert n_pos == n_shipped["pos"] == 79690 and n_nopos == n_shipped["nopos"] == 77290
print(f"phần của các khối Transformer: {pct(parts[f'{L} khối Transformer'] / n_pos, 1)}")

95,2% tham số nằm trong 4 khối Transformer; $E$ chỉ có 816. Bây giờ đến phép tính, so với hai mô hình của các bài
trước: LeNet của Bài 12 và MLP 784 → 128 → 64 → 10 của Bài 01–10.

In [ ]:
lenet_params = (5 * 5 * 1 * 6 + 6) + (5 * 5 * 6 * 16 + 16) + (400 * 120 + 120) + (120 * 84 + 84) + (84 * 10 + 10)
lenet_macs = 28 * 28 * 25 * 6 + 10 * 10 * 150 * 16 + 400 * 120 + 120 * 84 + 84 * 10
mlp_params = (784 * 128 + 128) + (128 * 64 + 64) + (64 * 10 + 10)
mlp_macs = 784 * 128 + 128 * 64 + 64 * 10
macs = {}
for size in (4, 7, 14):
    att, rest = vit_macs(size, D, L, MLP)
    macs[size] = att + rest
    print(
        f"ViT mảnh {size:>2}×{size:<2} ({(28 // size) ** 2 + 1:>2} token): {vn(vit_params(size, D, L, MLP)[1], 0):>6} tham số, "
        f"{vn(att + rest, 0):>9} MAC, chú ý chiếm {pct(att / (att + rest), 1)}"
    )
print(f"LeNet (Bài 12):              {vn(lenet_params, 0):>6} tham số, {vn(lenet_macs, 0):>9} MAC")
print(f"MLP 784 → 128 → 64 → 10:     {vn(mlp_params, 0):>6} tham số, {vn(mlp_macs, 0):>9} MAC")
assert (macs[4], macs[7], macs[14], lenet_macs, mlp_macs) == (4684512, 1402464, 416352, 416520, 109184)
assert (lenet_params, mlp_params) == (61706, 109386)

Mảnh 7×7 có đúng cùng số tham số với 4×4 (chỉ $E$ đổi cỡ, mà $E$ nhỏ), nhưng 16 token thay vì 49 nên chưa tốn tới một phần ba.
Mảnh 14×14 chỉ có 4 mảnh và tốn ngang LeNet. ViT với mảnh 4×4 tốn hơn 11 lần LeNet cho một chữ số 28×28: phép
chiếu được áp cho từng token, nên càng nhiều token càng đắt.

Cùng hai hàm đó đếm được ViT thật. ViT‑Ti/16 (Touvron và cộng sự, 2020: $D = 192$, 12 khối × 3 đầu, $M = 768$)
trên ảnh màu 224×224 với 1.000 lớp, so với số tham số trong tệp trọng số thật mà script của demo đã đếm:

In [ ]:
parts_ti, n_ti = vit_params(16, 192, 12, 768, img=224, C=3, classes=1000)
att_ti, rest_ti = vit_macs(16, 192, 12, 768, img=224, C=3, classes=1000)
print(f"ViT-Ti/16: {vn(n_ti, 0)} tham số theo công thức, {vn(pre['model']['params'], 0)} trong tệp trọng số thật")
print(f"{vn((att_ti + rest_ti) / 1e9, 2)} tỉ MAC mỗi ảnh, chú ý chiếm {pct(att_ti / (att_ti + rest_ti), 1)}")
assert n_ti == pre["model"]["params"] == 5717416

Công thức khớp đến từng tham số. Ở 197 token, phần chú ý chỉ chiếm khoảng một phần bảy phép tính; phần lớn vẫn là
phép nhân ma trận trên từng token.

## 7 · Chạy hai ViT đã huấn luyện

Hai mô hình dưới giống hệt nhau về kiến trúc và cách huấn luyện (55.000 ảnh, AdamW, 8.000 bước, seed 0), chỉ khác
một điều: `pos` có cộng vectơ vị trí học được, `nopos` thì không. Trước hết ta kiểm tra rằng hàm `forward` của ta
tính đúng như script đã huấn luyện chúng: so 10 logit của ảnh test đầu tiên (một số 7) với giá trị mà script in ra
khi xuất mô hình, rồi đếm số ảnh đúng trong 64 ảnh của demo.

In [ ]:
REF = {  # logit của ảnh test #0, do reel/vit/mnist.py tính bằng float64 khi xuất mô hình (check.json)
    "pos": [-1.9142892619, -0.7331674863, -0.4271250327, 1.7714222114, -3.7937051965,
            -0.0626626205, -2.9580504745, 9.6289179084, -2.0023079834, 0.3459495349],
    "nopos": [-6.4796454084, 1.7445987811, 1.4421554203, -1.6455837039, 1.0693166588,
              -1.3073301918, -2.3708101102, 8.3913805659, -5.1736052158, 2.5766033003],
}  # fmt: skip
X64 = digits / 255.0
z64 = {name: forward(p, X64) for name, p in vit64.items()}
for name in vit64:
    gap = np.abs(z64[name][0] - REF[name]).max()
    ok = int((z64[name].argmax(1) == y64).sum())
    print(f"{name:>5}: logit ảnh #0 = [{', '.join(vn(v, 2) for v in z64[name][0])}]")
    print(f"       lệch tối đa so với script {sci(gap)}; đúng {ok}/64 ảnh của demo")
    assert gap < 1e-8
assert [int((z64[n].argmax(1) == y64).sum()) for n in ("pos", "nopos")] == [64, 61]

Logit khớp với script đến từng chữ số thập phân thứ 10. Bây giờ chạy cả 10.000 ảnh test bằng bản float32 (nhanh hơn,
mỗi mô hình vài giây), rồi so với độ chính xác mà script đo cho seed 0 và khoảng giữa 3 seed.

In [ ]:
def group(gid, n=55000):
    """Một cấu hình trong tệp kết quả (id mô hình, số ảnh huấn luyện); g["seeds"] là 3 seed của nó."""
    return next(g for g in runs["groups"] if g["id"] == gid and g["n"] == n)


def span(v, digits=2):
    return f"{pct(min(v), digits)}–{pct(max(v), digits)}"


pred = {name: predict(p, Xte) for name, p in vit32.items()}
acc = {name: (pr == yte).mean() for name, pr in pred.items()}
for name, gid in (("pos", "vit4"), ("nopos", "vit4-nopos")):
    seeds = [s["test"] for s in group(gid)["seeds"]]
    print(
        f"{name:>5}: đúng {pct(acc[name])} trên 10.000 ảnh test; script: {pct(seeds[0])} (seed 0), {span(seeds)} (3 seed)"
    )
    assert abs(acc[name] - seeds[0]) < 3e-4, "khác với demo: hãy kiểm tra lại dữ liệu"
print(f"bỏ vectơ vị trí: mất {vn(100 * (acc['pos'] - acc['nopos']), 1)} điểm phần trăm")

Hai mô hình chỉ khác nhau 2.400 tham số vị trí, mà một bên đúng 97,83%, bên kia 89,07%. Mục sau tìm hiểu vì sao.

## 8 · Xáo trộn mảnh: vectơ vị trí làm gì?

Tự chú ý so từng cặp token bằng nội dung của chúng ($q_i \cdot k_j$) và cộng các $v_j$ lại. Không có gì trong phép
tính đó cho biết token $j$ nằm ở đâu trong ảnh. Thông tin vị trí duy nhất là $E_{\text{pos}}$ được cộng vào lúc đầu.

### 🤔 Dự đoán trước
Giữ nguyên nội dung từng mảnh, nhưng xáo thứ tự 49 mảnh (mỗi ảnh một cách xáo). Trong 64 ảnh của demo, mô hình không
vị trí (lúc chưa xáo đúng 61 ảnh) sẽ đúng bao nhiêu ảnh? Còn mô hình có vị trí (lúc chưa xáo đúng 64 ảnh)?

In [ ]:
def unpatchify(xp, P=P):
    """Ngược với patchify: (B, N, P²) → (B, 28, 28)."""
    B, G = len(xp), 28 // P
    return xp.reshape(B, G, G, P, P).transpose(0, 1, 3, 2, 4).reshape(B, 28, 28)


def shuffle_patches(X, orders, P=P):
    """Ô k của lưới nhận mảnh orders[b, k]: mỗi ảnh b một hoán vị riêng."""
    return unpatchify(patchify(X, P)[np.arange(len(X))[:, None], orders], P)


rng = np.random.default_rng(11)  # đúng các hoán vị mà script dùng để kiểm tra demo
Xs64 = shuffle_patches(X64, np.array([rng.permutation(49) for _ in range(64)]))
REF_SHUF = [-2.0778332685, 3.0591328714, -0.183120002, 0.6325104861, 1.1377082091,
            0.3617171861, -2.8125161098, 2.7310975398, -1.2453949321, -1.0833118026]  # fmt: skip
zs64 = {name: forward(p, Xs64) for name, p in vit64.items()}
for name in vit64:
    ok = int((zs64[name].argmax(1) == y64).sum())
    same = int((zs64[name].argmax(1) == z64[name].argmax(1)).sum())
    diff = np.abs(zs64[name] - z64[name]).max()
    print(f"{name:>5}: đúng {ok}/64 sau khi xáo; {same}/64 dự đoán như cũ; logit lệch tối đa {sci(diff)}")
assert np.abs(zs64["pos"][0] - REF_SHUF).max() < 1e-8  # ảnh #0 sau khi xáo, như script tính
assert np.abs(zs64["nopos"] - z64["nopos"]).max() < 1e-10
assert [int((zs64[n].argmax(1) == y64).sum()) for n in ("pos", "nopos")] == [19, 61]

Mô hình không vị trí cho **đúng cùng logit**; chênh lệch $10^{-15}$ chỉ là do cộng theo thứ tự khác. Không có
$E_{\text{pos}}$, mỗi token chỉ phụ thuộc nội dung mảnh của nó. LayerNorm, Q/K/V và MLP áp riêng cho từng token, còn
phép chú ý là tổng có trọng số trên *tập* các token. Vì vậy đổi chỗ các token đầu vào thì các token đầu ra đổi chỗ y
như thế (tính *đẳng biến hoán vị*). Token [CLS] không phải mảnh nên đứng yên, và đầu ra của nó gom từ mọi mảnh, nên
không đổi. Mô hình này chỉ thấy một **túi mảnh**.

Mô hình có vị trí thì sụp: từ 64/64 còn 19/64. Nó đã học “mảnh ở vị trí $k$ có nội dung này”, nên khi mảnh bị dời, nó
đọc ra một chữ số khác. Trên 500 ảnh test đầu tiên, với đúng các hoán vị mà script dùng khi đo cả tập test
(`default_rng(7)`, mỗi ảnh một hoán vị):

In [ ]:
N_SUB = 500  # ảnh test đầu tiên; script đo cả 10.000 ảnh
rng = np.random.default_rng(7)
X_patch = shuffle_patches(Xte[:N_SUB], np.array([rng.permutation(49) for _ in range(N_SUB)]))
pred_patch = {name: predict(p, X_patch) for name, p in vit32.items()}
for name, gid in (("pos", "vit4"), ("nopos", "vit4-nopos")):
    a_sub = (pred_patch[name] == yte[:N_SUB]).mean()
    stored = [s["patches"]["acc"] for s in group(gid)["seeds"]]
    print(
        f"{name:>5}: xáo mảnh, đúng {pct(a_sub, 1)} trong {vn(N_SUB, 0)} ảnh; cả 10.000 ảnh (script, 3 seed): {span(stored)}"
    )
same_nopos = int((pred_patch["nopos"] == pred["nopos"][:N_SUB]).sum())
print(f"nopos: {same_nopos}/{N_SUB} dự đoán như trên ảnh gốc")
assert same_nopos >= N_SUB - 1  # túi mảnh (float32: một hòa gần tuyệt đối có thể lật do làm tròn)

Mô hình có vị trí chỉ còn 18,76–21,01% trên cả tập test (3 seed), chỉ gấp đôi mức đoán mò (1 trên 10). Vậy mô hình
MNIST *dùng* vị trí rất nhiều. Mục 9 sẽ cho thấy nó dùng vị trí theo một cách bất ngờ.

### 🤔 Dự đoán trước
Bây giờ làm ngược lại: giữ mỗi mảnh ở đúng chỗ, nhưng xáo 16 điểm ảnh *bên trong* mỗi mảnh (mỗi ảnh một hoán vị, dùng
chung cho mọi mảnh của ảnh đó). Mô hình nào chịu được tốt hơn: có vị trí hay không vị trí?

In [ ]:
def shuffle_pixels(X, perms, P=P):
    """Xáo P² điểm ảnh bên trong mọi mảnh của ảnh b theo cùng một hoán vị perms[b]."""
    return unpatchify(np.take_along_axis(patchify(X, P), perms[:, None, :], axis=2), P)


rng = np.random.default_rng(7)  # như script
X_pix = shuffle_pixels(Xte[:N_SUB], np.array([rng.permutation(P * P) for _ in range(N_SUB)]))
pred_pix = {name: predict(p, X_pix) for name, p in vit32.items()}
for name, gid in (("pos", "vit4"), ("nopos", "vit4-nopos")):
    a_sub = (pred_pix[name] == yte[:N_SUB]).mean()
    stored = [s["pixels"]["acc"] for s in group(gid)["seeds"]]
    print(
        f"{name:>5}: xáo điểm ảnh, đúng {pct(a_sub, 1)} trong {vn(N_SUB, 0)} ảnh; cả 10.000 ảnh (script): {span(stored)}"
    )
ink_kept = np.allclose(patchify(X_pix, P).sum(2), patchify(Xte[:N_SUB], P).sum(2), atol=1e-5)
print("lượng mực của từng mảnh giữ nguyên sau khi xáo điểm ảnh:", ink_kept)
assert ink_kept

Mô hình có vị trí chịu được tốt hơn nhiều: 65,06–66,60% so với 11,79–13,67% (cả tập test, 3 seed). Mô hình không vị
trí chỉ có hình dạng nét bên trong từng mảnh để dựa vào, nên xáo điểm ảnh trong mảnh là mất gần hết. Mô hình có vị trí
còn một nguồn thông tin khác: mảnh ở vị trí nào có bao nhiêu mực, và xáo điểm ảnh giữ nguyên lượng mực đó (ô trên in
`True`).

Thử bỏ hẳn hình dạng nét: thay mỗi mảnh bằng một màu xám đều, đậm bằng trung bình của mảnh đó. Ảnh chỉ còn là một
lưới 7×7 điểm ảnh thô, và mô hình không vị trí chỉ thấy một túi 49 ô xám.

In [ ]:
coarse = unpatchify(np.repeat(patchify(Xte[:N_SUB], P).mean(2, keepdims=True), P * P, axis=2))
for name, p in vit32.items():
    print(f"{name:>5}: chỉ còn lưới 7×7 độ đậm, đúng {pct((predict(p, coarse) == yte[:N_SUB]).mean(), 1)}")

fig, axes = plt.subplots(1, 4, figsize=(8, 2.6))
k = 1  # ảnh test #1, một số 2
for ax, (im, title) in zip(
    axes,
    [
        (Xte[k], "ảnh gốc\n"),
        (X_patch[k], "xáo thứ tự\nmảnh"),
        (X_pix[k], "xáo điểm ảnh\ntrong mảnh"),
        (coarse[k], "chỉ giữ độ đậm\nmỗi mảnh"),
    ],
):
    ax.imshow(im, cmap="gray_r", vmin=0, vmax=1)
    ax.set_title(title, fontsize=9)
    ax.axis("off")
plt.show()

Hình trên là bốn phiên bản của một số 2: ảnh gốc, xáo mảnh, xáo điểm ảnh trong mảnh, và chỉ giữ độ đậm trung bình của
mỗi mảnh. Với mắt người, ảnh thứ hai là một mớ nét rời, còn ảnh cuối vẫn lờ mờ ra số 2. Mô hình có vị trí đọc giống
vậy: chỉ với lưới 7×7 độ đậm, nó vẫn đúng khoảng bảy phần mười số ảnh. Phần lớn những gì nó biết nằm ở *chỗ nào có
mực*, chứ không ở hình dạng nét bên trong mỗi mảnh.

## 9 · Vectơ vị trí học được gì?

Bài báo ViT vẽ cos giữa vectơ vị trí của từng cặp mảnh (Hình 7): sau huấn luyện, mảnh gần nhau có vectơ giống nhau
hơn, và mảnh cùng hàng hay cùng cột cũng vậy. Mô hình tự học ra lưới 2‑D, dù chỉ được cho 196 vectơ xếp thành một dãy.

### 🤔 Dự đoán trước
Mô hình MNIST có vị trí sụp khi xáo mảnh, nên nó dựa nhiều vào vị trí. Vậy vectơ vị trí của nó có học ra lưới 2‑D
không: cos giữa hai mảnh kề nhau có lớn hơn rõ rệt cos giữa hai mảnh cách xa nhau không?

In [ ]:
def grid_stats(pos, G):
    """cos giữa vectơ vị trí của mọi cặp mảnh (lưới G×G), và các số đo mà script của demo dùng."""
    pn = pos / np.linalg.norm(pos, axis=1, keepdims=True)
    cos = pn @ pn.T
    r, c = np.divmod(np.arange(G * G), G)
    dist = np.hypot(r[:, None] - r[None], c[:, None] - c[None])  # khoảng cách giữa hai mảnh, tính bằng số mảnh
    off = ~np.eye(G * G, dtype=bool)
    best = np.where(off, cos, -9).argmax(1)  # mảnh có vectơ vị trí giống nhất (trừ chính nó)
    return cos, {
        "near": cos[dist == 1].mean(),  # hai mảnh kề cạnh
        "far": cos[dist >= 4].mean(),  # hai mảnh cách ≥ 4 mảnh
        "corr": np.corrcoef(cos[off], -dist[off])[0, 1],  # tương quan giữa cos và độ gần
        "nearest": int((np.maximum(abs(r - r[best]), abs(c - c[best])) <= 1).sum()),  # giống nhất là mảnh kề
    }


cos_m, st = grid_stats(vit64["pos"]["pos"][1:], 7)  # bỏ vị trí của [CLS]
A_pos = runs["analysis"]["pos"]
print(
    f"ViT MNIST: cos hai mảnh kề {vn(st['near'], 3)}, cách ≥ 4 mảnh {vn(st['far'], 3)}; "
    f"tương quan cos–độ gần {vn(st['corr'], 3)}; mảnh giống nhất là mảnh kề: {st['nearest']}/49"
)
assert abs(st["near"] - A_pos["near"]) < 1e-3 and abs(st["corr"] - A_pos["corr"]) < 1e-3
assert st["nearest"] == A_pos["nearestNeighbour"] == 20
ever_ink = patchify(train_img[:10000] / 255.0, P).mean((0, 2)) >= 1e-3  # bỏ các mảnh viền hầu như luôn trống
cos_ink = cos_m[np.ix_(ever_ink, ever_ink)]
ri, ci = np.divmod(np.flatnonzero(ever_ink), 7)
d_ink = np.hypot(ri[:, None] - ri[None], ci[:, None] - ci[None])
print(
    f"chỉ xét {int(ever_ink.sum())} vị trí từng có nét: cos hai mảnh kề {vn(cos_ink[d_ink == 1].mean(), 3)}, "
    f"tương quan cos–độ gần {vn(np.corrcoef(cos_ink[d_ink > 0], -d_ink[d_ink > 0])[0, 1], 3)}"
)

Không. Cos giữa hai mảnh kề nhau chỉ 0,138, gần như bằng cos giữa hai mảnh cách xa nhau, và tương quan giữa cos với độ
gần của hai mảnh gần bằng 0. Kết quả không đổi khi bỏ 7 mảnh viền hầu như luôn trống (dòng thứ hai: chỉ xét 42 vị trí
từng có nét). Mô hình MNIST dùng vị trí rất nhiều (xáo mảnh là sụp) mà không học ra lưới: mỗi vectơ vị trí
giống một *số nhà* riêng, không phải một tọa độ. Mô hình học “mực ở ô số 24 nghĩa là gì”, như MLP học từng điểm ảnh,
mà không biết ô 24 nằm cạnh ô 25. Chữ số MNIST luôn được căn giữa, nên học từng vị trí riêng rẽ như vậy là đủ.

ViT‑Ti/16 thật (tiền huấn luyện trên ImageNet‑21k, 14 triệu ảnh, rồi tinh chỉnh trên ImageNet‑1k) thì khác. Script của
demo đã đo cos giữa 196 vectơ vị trí của nó; tệp `vit-pretrained.json` lưu cả ma trận 196 × 196 (mỗi số làm tròn về
một byte) và các số đo:

In [ ]:
cos_t = np.frombuffer(base64.b64decode(pre["pos"]["cos"]), np.uint8).reshape(196, 196) / 255 * 2 - 1
pt = pre["pos"]
print(
    f"ViT-Ti/16: cos hai mảnh kề {vn(pt['near'], 3)}; cùng hàng hoặc cùng cột, cách ≥ 4 mảnh {vn(pt['lineFar'], 3)}; "
    f"khác hàng khác cột, cách ≥ 4 mảnh {vn(pt['offFar'], 3)}"
)
print(f"tương quan cos–độ gần {vn(pt['corr'], 3)}; mảnh giống nhất là mảnh kề: {pt['nearestNeighbour']}/196")

fig, axes = plt.subplots(1, 4, figsize=(10, 2.9))
for ax, (cos, G_, sel, name) in zip(
    axes,
    [
        (cos_m, 7, 24, "ViT MNIST"),
        (cos_m, 7, 12, "ViT MNIST"),
        (cos_t, 14, 65, "ViT‑Ti/16"),
        (cos_t, 14, 142, "ViT‑Ti/16"),
    ],
):
    im = ax.imshow(cos[sel].reshape(G_, G_), cmap=DIVERGING, vmin=-1, vmax=1)
    r_, c_ = divmod(sel, G_)
    ax.add_patch(Rectangle((c_ - 0.5, r_ - 0.5), 1, 1, fc="none", ec="black", lw=1.5))
    ax.set_title(f"{name}: mảnh hàng {r_ + 1}, cột {c_ + 1}", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(im, ax=axes, fraction=0.02, label="cos với mảnh đã chọn")
vn_axes(fig)
plt.show()

Mỗi ô là cos giữa vectơ vị trí của mảnh được đóng khung đen và vectơ vị trí của mảnh ở ô đó (xanh: giống, cam: ngược
chiều). Hai hình bên trái (MNIST, lưới 7×7) gần như là nhiễu: ô kề với ô đã chọn không xanh hơn các ô khác. Hai hình
bên phải (ViT‑Ti, lưới 14×14) cho thấy một vệt xanh quanh mảnh đã chọn, kéo dài theo hàng và cột của nó. Không ai cho
ViT‑Ti biết ảnh là một lưới 14×14; nó tự học ra điều đó từ dữ liệu, vì trên ảnh thật vật thể có thể nằm ở bất cứ đâu và
hai mảnh kề nhau thường thuộc cùng một vật.

Bài báo ViT cũng thử đưa vị trí 2‑D vào sẵn (Bảng 8, ViT‑B/16 tiền huấn luyện rồi đánh giá trên ImageNet bằng phân loại
tuyến tính với 5 ảnh mỗi lớp): không vị trí 61,4%, vị trí 1‑D học được 64,2%, vị trí 2‑D 64,0%, vị trí tương đối
64,0%. Có vị trí thì quan trọng, còn kiểu vị trí nào thì gần như không.

### Attention của ViT MNIST nhìn đâu?

Hình dưới vẽ attention của token [CLS] tới 49 mảnh ở từng khối (trung bình 3 đầu) cho ảnh test đầu tiên, một số 7.
Màu xanh càng đậm, [CLS] càng lấy nhiều thông tin từ mảnh đó.

In [ ]:
_, att0 = forward(vit64["pos"], X64[:1], maps=True)
BLUES = LinearSegmentedColormap.from_list("trang_xanh", ["#ffffff", BLUE])
fig, axes = plt.subplots(1, L, figsize=(9, 2.6))
for l, ax in enumerate(axes):
    m = att0[l][0, :, 0, 1:].mean(0).reshape(7, 7)  # [CLS] → 49 mảnh, trung bình 3 đầu
    ax.imshow(m, cmap=BLUES, vmin=0, extent=(-0.5, 27.5, 27.5, -0.5))
    ax.contour(X64[0], levels=[0.5], colors=ORANGE, linewidths=1.2)  # nét chữ số (cam)
    ax.set_title(f"khối {l + 1}: {pct(m.max(), 0)} vào 1 mảnh", fontsize=9)
    ax.axis("off")
plt.show()
inked = patchify(X64[:1], P)[0].max(1) > 0  # mảnh có nét
cls_maps = [att0[l][0, :, 0, 1:].mean(0) for l in range(L)]  # [CLS] → 49 mảnh, trung bình 3 đầu
print(f"mảnh nhận nhiều nhất, từng khối: {', '.join(pct(m.max(), 0) for m in cls_maps)} (chia đều: {pct(1 / 49, 0)})")
print(
    f"{int(inked.sum())}/49 mảnh có nét ({pct(inked.mean(), 0)}) nhận: "
    + ", ".join(pct(m[inked].sum() / m.sum(), 0) for m in cls_maps)
)
print("[CLS] nhìn chính nó, từng khối:", ", ".join(pct(att0[l][0, :, 0, 0].mean()) for l in range(L)))

Đường cam là viền nét chữ. Ở mỗi khối, [CLS] dồn attention vào vài mảnh: mảnh nhận nhiều nhất được 14–34%, trong khi
chia đều thì mỗi mảnh chỉ được 2%. Ở khối 1 và 4, các mảnh đậm nhất nằm dọc thân số 7. Nhưng 16 mảnh có nét (33% số
mảnh) chỉ nhận 39–53% attention: [CLS] cũng đọc nhiều từ các mảnh nền, và gần như không nhìn chính nó. Một bản đồ
attention chỉ cho biết thông tin đi từ đâu tới [CLS] ở một khối, chưa phải lý do của dự đoán: các mảnh đã trộn thông
tin của nhau ở các khối trước.

Để đo xem một đầu nhìn gần hay nhìn xa, bài báo ViT dùng **khoảng cách chú ý trung bình**: với mỗi mảnh, lấy trung bình
khoảng cách (px, giữa tâm hai mảnh) tới các mảnh nó nhìn, có trọng số là attention. Nó giống kích thước vùng tiếp
nhận của CNN (Bài 12). Ô dưới đo trên 1.000 ảnh test, như script của demo. Nếu một đầu nhìn đều khắp ảnh, con số là
khoảng cách trung bình giữa hai mảnh bất kỳ.

In [ ]:
G = 28 // P
r, c = np.divmod(np.arange(G * G), G)
dist_px = np.hypot(r[:, None] - r[None], c[:, None] - c[None]) * P  # px giữa tâm hai mảnh


def attention_distance(p, X, bs=250):
    """(L, đầu): khoảng cách chú ý trung bình, chỉ xét mảnh → mảnh (bỏ [CLS], chia lại cho tổng bằng 1)."""
    out = np.zeros((L, HEADS))
    for i in range(0, len(X), bs):
        for l, A in enumerate(forward(p, X[i : i + bs], maps=True)[1]):
            pp = A[:, :, 1:, 1:].astype(np.float64)
            pp /= pp.sum(-1, keepdims=True)
            out[l] += (pp * dist_px).sum(-1).mean(-1).sum(0)
    return out / len(X)


N_ATT = runs["setup"]["nAtt"]  # 1.000 ảnh test đầu tiên, như script
dist_mnist = {name: attention_distance(p, Xte[:N_ATT]) for name, p in vit32.items()}
uniform = dist_px.mean()
print(f"nhìn đều khắp ảnh: {vn(uniform, 1)} px")
for name, d in dist_mnist.items():
    print(f"{name:>5}: " + " · ".join(f"khối {l + 1}: " + ", ".join(vn(v, 1) for v in d[l]) for l in range(L)))
    assert np.abs(d - np.array(runs["analysis"][name]["dist"])).max() < 2e-3, "khác với demo"
assert abs(uniform - runs["analysis"]["pos"]["uniform"]) < 1e-3

Mọi đầu, ở mọi khối, của cả hai mô hình đều nhìn xa 12–15 px, gần bằng mức nhìn đều khắp ảnh (14,4 px). Không đầu nào
chuyên nhìn quanh mảnh của mình như một bộ lọc tích chập. Với 55.000 chữ số nhỏ được căn giữa, mô hình không cần
những đầu như vậy để đạt 97,83%.

### ViT‑Ti/16 thật: đầu nhìn gần, đầu nhìn xa

Bài báo ViT (Hình 7, phải) thấy điều khác ở các mô hình lớn huấn luyện trên nhiều dữ liệu: ngay ở khối đầu, vài đầu
nhìn khắp ảnh, còn vài đầu khác chỉ nhìn quanh mảnh của mình, giống tích chập; càng sâu thì mọi đầu càng nhìn xa.
Raghu và cộng sự (2021) thấy rằng khi chỉ học trên ImageNet, các lớp đầu của ViT‑L/16 và ViT‑H/14 không học được những
đầu nhìn gần ấy. Script của demo đã đo đúng số này trên ViT‑Ti/16 với 8 ảnh chụp (224×224, mảnh 16 px). Hình dưới vẽ nó
từ tệp `vit-pretrained.json`: mỗi đầu một dấu (tròn, vuông, tam giác), đường gạch xám là mức nhìn đều khắp ảnh.

In [ ]:
dist_ti, dist_ti_clean = np.array(pre["dist"]), np.array(pre["distClean"])  # (12 khối, 3 đầu), trung bình 8 ảnh
MARKS = [("o", BLUE), ("s", ORANGE), ("^", GREEN)]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.3), sharey=True)
for ax, d, title in (
    (axes[0], dist_ti, "mọi mảnh"),
    (axes[1], dist_ti_clean, "bỏ các mảnh có chuẩn lớn (mục dưới)"),
):
    for h, (mk, col) in enumerate(MARKS):
        ax.plot(np.arange(1, 13), d[:, h], marker=mk, color=col, lw=1.2, ms=6, label=f"đầu {h + 1}")
    ax.axhline(pre["uniform"], color=GREY, ls="--", lw=1)
    ax.set(xlabel="khối", title=f"ViT‑Ti/16, {title}", xticks=range(1, 13), ylim=(0, 160))
    ax.grid(axis="y", alpha=0.3)
axes[0].set_ylabel("khoảng cách chú ý (px)")
axes[0].legend(frameon=False, fontsize=9, loc="lower right")
vn_axes(fig)
plt.show()
print(f"nhìn đều khắp ảnh 224×224: {vn(pre['uniform'], 1)} px")
print("khối 1, ba đầu:", ", ".join(f"{vn(v, 0)} px" for v in dist_ti[0]))
print("khối 12, ba đầu:", ", ".join(f"{vn(v, 0)} px" for v in dist_ti[11]))
print("khối 12, bỏ mảnh chuẩn lớn:", ", ".join(f"{vn(v, 0)} px" for v in dist_ti_clean[11]))

Ở khối 1, đầu 3 chỉ nhìn trong bán kính khoảng 26 px (chưa tới hai mảnh), còn đầu 2 nhìn xa 81 px. Càng sâu, cả ba đầu
càng nhìn xa. Từ khối 10 (hình trái), cả ba đầu còn nhìn xa hơn mức nhìn đều khắp ảnh: attention dồn về vài mảnh ở
rất xa. Bỏ những mảnh đó ra (hình phải) thì khoảng cách ở khối 12 về lại quanh 115 px. Đó là những mảnh nào?

### Bốn góc ảnh hút attention

Script đo chuẩn (độ dài) của từng token mảnh sau mỗi khối, và đánh dấu token nào có chuẩn lớn hơn 5 lần trung vị.
Ô dưới đọc kết quả cho 8 ảnh: lớp mà mô hình đoán, các mảnh chuẩn lớn ở khối 12 (mảnh 0 là góc trên trái, 13 góc trên
phải, 182 góc dưới trái, 195 góc dưới phải), và phần attention mà [CLS] dành cho chúng ở khối 12.

In [ ]:
CORNERS = {0, 13, 182, 195}
first_high, mass, ratios = [], [], []
for ph in pre["photos"]:
    norms = np.frombuffer(base64.b64decode(ph["norms"]), np.uint8).reshape(12, 196) / 255 * ph["normsMax"]
    ratios.append(norms[11].max() / ph["normMedian"][11])
    first_high.append(next(l for l, hs in enumerate(ph["high"]) if hs) + 1)
    mass.append(ph["highMassCls"][11])
    k_top, p_top = ph["top"][0]
    verdict = "đúng" if k_top in ph["expect"] else ("không có lớp đúng" if not ph["expect"] else "sai")
    print(
        f"{ph['label']:<22} đoán “{pre['classes'][str(k_top)]}” {pct(p_top, 1):>6} ({verdict}); "
        f"mảnh chuẩn lớn ở khối 12: {ph['high'][11]}, [CLS] dành cho chúng {pct(ph['highMassCls'][11], 1)}"
    )
print(f"mảnh chuẩn lớn đầu tiên xuất hiện ở khối {sorted(set(first_high))}; từ đó đến khối 12 chúng luôn là góc ảnh:",
      all(set(h) <= CORNERS for ph in pre["photos"] for h in ph["high"]))  # fmt: skip
print(f"ở khối 12, chuẩn lớn nhất gấp {vn(min(ratios), 0)}–{vn(max(ratios), 0)} lần trung vị")
print(
    f"[CLS] ở khối 12 dành {pct(min(mass), 0)}–{pct(max(mass), 0)} attention cho 3–4 mảnh góc đó (4 mảnh = {pct(4 / 196, 0)} số mảnh)"
)

Ở cả 8 ảnh, bất kể ảnh chụp chó, xe buýt hay tách cà phê, từ khối 8 trở đi có 3–4 token ở đúng các góc ảnh với chuẩn
lớn gấp hàng chục lần các token khác. Ở khối 12, [CLS] dành cho 3–4 mảnh đó từ 22% đến 53% attention của nó, dù chúng
chỉ là 2% số mảnh và gần như không chứa thông tin về vật trong ảnh. Hình dưới cho thấy điều đó trên hai ảnh: ảnh gốc,
attention của [CLS] ở khối 4 và khối 12 (trung bình 3 đầu; khung cam gạch là mảnh chuẩn lớn), và *attention rollout*
(Abnar & Zuidema, 2020: nhân dồn attention qua 12 khối, có cộng thêm kết nối tắt).

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(10, 5.3))
for row, i in enumerate([0, 2]):  # chó golden retriever, xe buýt trường học
    ph = pre["photos"][i]
    cls = np.frombuffer(base64.b64decode(ph["cls"]), np.uint8).reshape(12, 3, 196) / 255
    cls = cls * np.array(ph["clsMax"])[:, :, None]  # mỗi bản đồ được lưu tương đối so với giá trị lớn nhất của nó
    roll = np.frombuffer(base64.b64decode(ph["rollout"]), np.uint8) / 255 * ph["rolloutMax"]
    axes[row, 0].imshow(plt.imread(io.BytesIO(base64.b64decode(ph["jpeg"])), format="jpeg"))
    axes[row, 0].set_title(ph["label"], fontsize=9)
    maps = [
        (cls[3].mean(0), 3, "[CLS] ở khối 4"),
        (cls[11].mean(0), 11, "[CLS] ở khối 12"),
        (roll, 11, "nhân dồn 12 khối"),
    ]
    for ax, (m, blk, title) in zip(axes[row, 1:], maps):
        ax.imshow(m.reshape(14, 14), cmap=BLUES, vmin=0, extent=(0, 224, 224, 0))
        for j in ph["high"][blk]:
            r_, c_ = divmod(j, 14)
            ax.add_patch(Rectangle((c_ * 16, r_ * 16), 16, 16, fc="none", ec=ORANGE, lw=1.8, ls="--"))
        ax.set_title(title, fontsize=9)
    for ax in axes[row]:
        ax.axis("off")
plt.tight_layout()
plt.show()

Ở khối 4, attention của [CLS] còn rải khắp ảnh và chưa có mảnh chuẩn lớn nào. Ở khối 12, gần như tất cả dồn vào các
góc (khung cam). Rollout trộn mọi khối: với xe buýt nó sáng lên trên thân xe, với con chó thì vẫn rải gần đều. Nó là
đường thông tin đi, không phải lời giải thích cho dự đoán.

Darcet và cộng sự (2023) gặp hiện tượng giống vậy ở các ViT lớn (DeiT‑III và OpenCLIP từ cỡ ViT‑B, DINOv2 từ cỡ ViT‑L): khoảng 2%
token, thường là các mảnh nền rất giống mảnh bên cạnh, có chuẩn lớn gấp khoảng 10 lần và mang thông tin toàn cục thay vì
thông tin của mảnh. Mô hình dùng những mảnh thừa thông tin làm chỗ chứa tạm cho phép tính của nó. Cách sửa của họ: thêm
vài token **thanh ghi** (register), không thuộc mảnh nào, để mô hình có chỗ riêng cho việc đó. Ý tưởng giống *bể chú ý*
(attention sink) của GPT‑2 ở Bài 17: softmax buộc mỗi hàng có tổng bằng 1, nên mô hình cần vài token làm chỗ đặt phần
attention không dùng đến. Ở ViT‑Ti nhỏ này, các token đó không chọn mảnh nền mà luôn nằm ở bốn góc.

## 10 · Tự huấn luyện một ViT nhỏ

Hai ViT ở trên học 8.000 bước trên máy nhiều lõi. Để thấy cơ chế học, ta viết lan truyền ngược cho chính hàm `forward`
ở mục 5 (cùng các bước như `reel/vit/vit.py`) và huấn luyện một ViT nhỏ hơn nhiều: mảnh 7×7 (16 token), $D = 32$,
2 khối × 2 đầu, AdamW, 400 bước × 64 ảnh, chưa tới nửa lượt qua 55.000 ảnh. Trước hết, kiểm tra gradient bằng sai phân
hữu hạn trên một mô hình tí hon (float64).

In [ ]:
def layer_norm_back(dy, g, cache):
    hn, inv = cache
    dhn = dy * g
    dx = inv * (dhn - dhn.mean(-1, keepdims=True) - hn * (dhn * hn).mean(-1, keepdims=True))
    return dx, (dy * hn).reshape(-1, hn.shape[-1]).sum(0), dy.reshape(-1, hn.shape[-1]).sum(0)


def backward(p, cache, dz, heads):
    """Gradient của mọi tham số, đi ngược đúng các bước của forward(..., cache=True)."""
    xp, saved, hf, cf = cache
    B, N, _ = xp.shape
    T, Dm = N + 1, p["We"].shape[1]
    dh = Dm // heads
    f = lambda a: a.reshape(-1, a.shape[-1])  # (B, T, ·) → (B·T, ·)
    g = {"Wh": hf.T @ dz, "bh": dz.sum(0)}
    dcls, g["gf"], g["bf"] = layer_norm_back(dz @ p["Wh"].T, p["gf"], cf)
    dH = np.zeros((B, T, Dm), dz.dtype)
    dH[:, 0] = dcls  # chỉ token [CLS] đi vào bộ phân loại
    for l in reversed(range(len(saved))):
        a, c1, q, k, v, A, o, m, c2, u, gu, t = saved[l]
        w = {kk.split(".", 1)[1]: vv for kk, vv in p.items() if kk.startswith(f"{l}.")}
        g[f"{l}.W2"], g[f"{l}.c2"] = f(gu).T @ f(dH), f(dH).sum(0)  # MLP
        du = (dH @ w["W2"].T) * (0.5 * (1 + t) + 0.5 * u * (1 - t * t) * C_GELU * (1 + 3 * 0.044715 * u * u))
        g[f"{l}.W1"], g[f"{l}.c1"] = f(m).T @ f(du), f(du).sum(0)
        dx, g[f"{l}.g2"], g[f"{l}.b2"] = layer_norm_back(du @ w["W1"].T, w["g2"], c2)
        dH = dH + dx
        g[f"{l}.Wo"], g[f"{l}.bo"] = f(o).T @ f(dH), f(dH).sum(0)  # chú ý
        do = (dH @ w["Wo"].T).reshape(B, T, heads, dh).transpose(0, 2, 1, 3)
        dA = do @ v.transpose(0, 1, 3, 2)
        ds = A * (dA - (dA * A).sum(-1, keepdims=True)) * (1 / dh**0.5)  # qua softmax
        dqkv = np.stack([ds @ k, ds.transpose(0, 1, 3, 2) @ q, A.transpose(0, 1, 3, 2) @ do])
        dqkv = dqkv.transpose(1, 3, 0, 2, 4).reshape(B, T, 3 * Dm)
        g[f"{l}.Wqkv"], g[f"{l}.bqkv"] = f(a).T @ f(dqkv), f(dqkv).sum(0)
        dx, g[f"{l}.g1"], g[f"{l}.b1"] = layer_norm_back(dqkv @ w["Wqkv"].T, w["g1"], c1)
        dH = dH + dx
    if "pos" in p:
        g["pos"] = dH.sum(0)
    g["cls"] = dH[:, 0].sum(0)
    g["We"], g["be"] = xp.reshape(-1, xp.shape[-1]).T @ f(dH[:, 1:]), f(dH[:, 1:]).sum(0)
    return g


def softmax_ce(z, y):
    e = np.exp(z - z.max(1, keepdims=True))
    pr = e / e.sum(1, keepdims=True)
    loss = -np.log(pr[np.arange(len(y)), y]).mean()
    pr[np.arange(len(y)), y] -= 1
    return loss, pr / len(y)


def init_vit(P, Dm, L, M, seed, pos=True, dtype=np.float32):
    """Như reel/vit/vit.py: ma trận ~ N(0, 1/số đầu vào), vị trí ~ N(0; 0,02²), [CLS] và bộ phân loại bằng 0."""
    rng = np.random.default_rng(seed)
    lin = lambda i, o: rng.normal(0, (1 / i) ** 0.5, (i, o)).astype(dtype)
    zeros, ones = (lambda n: np.zeros(n, dtype)), (lambda n: np.ones(n, dtype))
    p = {"We": lin(P * P, Dm), "be": zeros(Dm), "cls": zeros(Dm)}
    if pos:
        p["pos"] = rng.normal(0, 0.02, ((28 // P) ** 2 + 1, Dm)).astype(dtype)
    for l in range(L):
        p |= {f"{l}.g1": ones(Dm), f"{l}.b1": zeros(Dm), f"{l}.Wqkv": lin(Dm, 3 * Dm), f"{l}.bqkv": zeros(3 * Dm)}
        p |= {f"{l}.Wo": lin(Dm, Dm), f"{l}.bo": zeros(Dm), f"{l}.g2": ones(Dm), f"{l}.b2": zeros(Dm)}
        p |= {f"{l}.W1": lin(Dm, M), f"{l}.c1": zeros(M), f"{l}.W2": lin(M, Dm), f"{l}.c2": zeros(Dm)}
    return p | {"gf": ones(Dm), "bf": zeros(Dm), "Wh": np.zeros((Dm, 10), dtype), "bh": zeros(10)}


rng = np.random.default_rng(0)
tiny = init_vit(7, 8, 2, 12, seed=1, dtype=np.float64)
for key in tiny:  # rời khỏi điểm khởi tạo đối xứng (bộ phân loại bằng 0, LayerNorm nhân 1)
    tiny[key] = tiny[key] + 0.3 * rng.standard_normal(tiny[key].shape)
Xg, yg = rng.random((3, 28, 28)), rng.integers(0, 10, 3)
z, cache = forward(tiny, Xg, P=7, heads=2, cache=True)
grads = backward(tiny, cache, softmax_ce(z, yg)[1], heads=2)
worst = 0.0
for key in tiny:
    for _ in range(3):
        idx = tuple(rng.integers(0, s) for s in tiny[key].shape)
        old = tiny[key][idx]
        tiny[key][idx] = old + 1e-5
        lp = softmax_ce(forward(tiny, Xg, P=7, heads=2), yg)[0]
        tiny[key][idx] = old - 1e-5
        lm = softmax_ce(forward(tiny, Xg, P=7, heads=2), yg)[0]
        tiny[key][idx] = old
        num = (lp - lm) / 2e-5
        worst = max(worst, abs(num - grads[key][idx]) / max(1e-5, abs(num) + abs(grads[key][idx])))
print(f"{len(tiny)} mảng tham số, mỗi mảng 3 phần tử: sai số tương đối lớn nhất {sci(worst)}")
assert worst < 1e-5

Gradient khớp với sai phân hữu hạn. Vòng huấn luyện dưới dùng AdamW (suy giảm trọng số 0,05 trên các ma trận), tăng
dần tốc độ học trong 5% số bước đầu rồi giảm theo cosin, như script của demo. Ta huấn luyện hai bản, có và không có
vectơ vị trí, cùng seed nên cùng lô ảnh, rồi đo trên 10.000 ảnh test.

In [ ]:
def train_vit(P=7, Dm=32, L=2, heads=2, M=64, steps=400, eta=4e-3, seed=0, pos=True, wd=0.05):
    p = init_vit(P, Dm, L, M, seed, pos)
    mom = {k: np.zeros_like(v) for k, v in p.items()}
    sq = {k: np.zeros_like(v) for k, v in p.items()}
    rng = np.random.default_rng(seed)
    warm = steps // 20
    for t in range(1, steps + 1):
        lr = eta * (t / warm if t <= warm else 0.5 * (1 + math.cos(math.pi * (t - warm) / (steps - warm))))
        idx = rng.integers(0, 55000, 64)  # lô 64 ảnh từ train[:55k]
        z, cache = forward(p, Xtr[idx], P=P, heads=heads, cache=True)
        g = backward(p, cache, softmax_ce(z, ytr[idx])[1].astype(np.float32), heads)
        for k in p:  # AdamW
            mom[k] = 0.9 * mom[k] + 0.1 * g[k]
            sq[k] = 0.999 * sq[k] + 0.001 * g[k] ** 2
            if k.split(".")[-1][0] == "W":  # suy giảm trọng số chỉ cho các ma trận (We, Wqkv, Wo, W1, W2, Wh)
                p[k] *= 1 - lr * wd
            p[k] -= lr * (mom[k] / (1 - 0.9**t)) / (np.sqrt(sq[k] / (1 - 0.999**t)) + 1e-8)
    return p


Xtr = (train_img / 255.0).astype(np.float32)
ytr = train_lab.astype(np.int64)
small = {pos: train_vit(pos=pos) for pos in (True, False)}
acc_small = {pos: (predict(p, Xte, P=7, heads=2) == yte).mean() for pos, p in small.items()}
n_small = sum(a.size for a in small[True].values())
print(f"ViT nhỏ (mảnh 7×7, D = 32, 2 khối, {vn(n_small, 0)} tham số), 400 bước:")
print(f"  có vị trí {pct(acc_small[True])}, không vị trí {pct(acc_small[False])} trên 10.000 ảnh test")
# 87,32% / 71,29% ở đây; seed 1–2 cho 89,33% / 76,56% và 88,47% / 75,99%. Làm tròn float32 khác trên x86 nên dung sai rộng
assert abs(acc_small[True] - 0.8732) < 0.03 and acc_small[True] - acc_small[False] > 0.05

Chỉ 400 bước, chưa tới nửa lượt dữ liệu, ViT nhỏ đã đọc đúng gần chín phần mười chữ số; bỏ vectơ vị trí thì mất hơn
chục điểm.
Mô hình ở mục 7 học 8.000 bước với $D = 48$ và 4 khối mới lên 97,83%. Câu hỏi còn lại: với cùng công thức huấn luyện,
ViT so với một CNN thì thế nào, và điều đó phụ thuộc vào lượng dữ liệu ra sao?

## 11 · ViT và CNN: cần bao nhiêu dữ liệu?

Script của demo huấn luyện ViT (mảnh 4×4 và 7×7), LeNet của Bài 12 và MLP với cùng một công thức: AdamW, 8.000 bước,
lô 64, tốc độ học $\eta$ chọn trên 5.000 ảnh kiểm định (`train[55k:]`, không bao giờ trên tập test). Mỗi cỡ tập huấn
luyện từ 1.000 đến 55.000 ảnh, 3 seed. Mọi cỡ đều có cùng số bước, nên tập nhỏ được xem lại nhiều lần hơn.

### 🤔 Dự đoán trước
Với 1.000 ảnh huấn luyện, LeNet hơn ViT mảnh 4×4 bao nhiêu điểm phần trăm? Đến 55.000 ảnh, ViT có vượt được LeNet không?

In [ ]:
SIZES = runs["setup"]["sizes"]
MODELS = [("vit4", "ViT mảnh 4×4", BLUE, "o"), ("vit7", "ViT mảnh 7×7", GREEN, "^"),
          ("lenet", "LeNet (CNN)", ORANGE, "s"), ("mlp", "MLP", GREY, "D")]  # fmt: skip


def tests(gid, n=55000):
    return [s["test"] for s in group(gid, n)["seeds"]]


med = {gid: np.array([np.median(tests(gid, n)) for n in SIZES]) for gid, *_ in MODELS}
print("số ảnh   " + "".join(f"{name:>15}" for _, name, *_ in MODELS) + "   LeNet − ViT 4×4")
for i, n in enumerate(SIZES):
    row = "".join(f"{pct(med[gid][i]):>15}" for gid, *_ in MODELS)
    print(f"{vn(n, 0):>7}  {row}   {vn(100 * (med['lenet'][i] - med['vit4'][i]), 1):>6} điểm")
bad = [s for s in group("mlp", 1000)["seeds"] if s["test"] < 0.5]
print(f"(MLP, 1.000 ảnh: seed {bad[0]['seed']} không học được, {pct(bad[0]['test'])}; bảng dùng trung vị 3 seed)")
tr = {gid: min(s["train"] for s in group(gid, 1000)["seeds"]) for gid in ("vit4", "lenet")}
print(f"đúng trên chính 1.000 ảnh huấn luyện (seed kém nhất): ViT {pct(tr['vit4'])}, LeNet {pct(tr['lenet'])}")

LeNet hơn ViT ở **mọi** cỡ dữ liệu. Khoảng cách thu hẹp từ 8,5 điểm ở 1.000 ảnh xuống 1,4 điểm ở 55.000 ảnh, nhưng
không đổi chiều. Ở 1.000 ảnh, cả hai đều đoán đúng hết các ảnh huấn luyện: cả hai đều thuộc lòng dữ liệu, và chỉ
khác nhau ở chỗ đoán ảnh mới. Hình dưới vẽ cùng bảng đó: mỗi dấu nhỏ là một seed, đường nối các trung vị, mỗi mô hình
một màu và một hình dấu riêng.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for gid, name, col, mk in MODELS:
    for i, n in enumerate(SIZES):
        v = np.array(tests(gid, n))
        v = v[v > 0.5]  # seed không học được của MLP nằm ngoài trục, xem dòng in ở trên
        ax.scatter([n] * len(v), 100 * v, color=col, marker=mk, s=14, alpha=0.5, lw=0)
    ax.plot(SIZES, 100 * med[gid], color=col, marker=mk, ms=7, lw=1.8, label=name)
ax.set(xscale="log", xlabel="số ảnh huấn luyện (thang log)", ylabel="đúng trên 10.000 ảnh test (%)", ylim=(84, 100))
ax.set_title("Cùng công thức huấn luyện, 3 seed mỗi điểm", fontsize=11)
ax.grid(alpha=0.3)
ax.legend(frameon=False, loc="lower right")
vn_axes(fig)
ax.set_xticks(SIZES, labels=[vn(n, 0) for n in SIZES])
plt.show()

Tích chập cài sẵn hai giả định mà ViT phải tự học: điểm ảnh gần nhau liên quan với nhau (tính cục bộ), và một nét ở đâu
cũng là nét đó (bộ lọc dùng chung khi trượt). Đó là **thiên kiến quy nạp** (inductive bias). Khi dữ liệu ít, những giả
định đúng giúp tổng quát hóa; khi dữ liệu nhiều, mô hình tự học được phần lớn chúng, nên khoảng cách nhỏ dần.

Cũng tệp đó có vài thí nghiệm phụ, mỗi cái với $\eta$ chọn riêng. Ta in khoảng giữa 3 seed và so với ViT mảnh 4×4 ở
cùng số ảnh; hai khoảng chồng nhau thì chênh lệch nằm trong dao động giữa các seed.

In [ ]:
def compare(gid, ref, n, n_ref=None, label=""):
    a, b = tests(gid, n), tests(ref, n_ref or n)
    if min(a) <= max(b) and min(b) <= max(a):
        verdict = "trong dao động giữa các seed"
    else:
        verdict = (
            f"{'hơn' if np.median(a) > np.median(b) else 'kém'} {vn(100 * abs(np.median(a) - np.median(b)), 1)} điểm"
        )
    print(f"{label:<44} {span(a):>15}  so với {span(b):>15}: {verdict}")


compare("vit7", "vit4", 55000, label="mảnh 7×7 (55.000 ảnh)")
compare("vit14", "vit4", 55000, label="mảnh 14×14, chỉ 4 mảnh (55.000 ảnh)")
compare("vit7", "vit4", 1000, label="mảnh 7×7 (1.000 ảnh)")
compare("vit4-gap", "vit4", 55000, label="gộp trung bình (GAP) thay cho [CLS]")
compare("vit4-nopos", "vit4", 55000, label="không vectơ vị trí")
compare("vit4-aug", "vit4", 5000, label="5.000 ảnh + dịch ngẫu nhiên ±2 px")
compare("vit4-aug", "vit4", 5000, 55000, label="5.000 ảnh + dịch ±2 px, so với ViT 55.000 ảnh")
compare("lenet-aug", "lenet", 5000, label="LeNet, 5.000 ảnh + dịch ±2 px")

Bốn điều đáng nhớ:
- **Mảnh nhỏ không tự động tốt hơn.** Trên chữ số 28×28, mảnh 7×7 và 14×14 đều hơn 4×4 ở 55.000 ảnh, mà rẻ hơn 3 và
  11 lần (mục 6). Ở 1.000 ảnh, 7×7 và 4×4 ngang nhau.
- **[CLS] không bắt buộc.** Lấy trung bình mọi mảnh (GAP) cho kết quả ngang [CLS], như bài báo ViT thấy (Phụ lục D.3),
  miễn là chọn lại tốc độ học.
- **Vị trí là quan trọng:** bỏ vectơ vị trí mất gần 9 điểm.
- **Tăng cường dữ liệu thay được một phần dữ liệu thật.** Chỉ dịch ngẫu nhiên mỗi ảnh tới 2 px, ViT học trên 5.000 ảnh
  đã ngang ViT học trên 55.000 ảnh. LeNet cũng hơn lên, nhưng ít hơn, vì tích chập đã có sẵn phần lớn tính bền với
  phép dịch.

Trên ảnh thật, bài báo ViT thấy cùng một xu hướng ở quy mô lớn hơn nhiều. Chỉ học trên ImageNet (1,3 triệu ảnh),
ViT‑B/16 đạt 77,9% (Bảng 5) và kém các ResNet (BiT) cùng cỡ. Tiền huấn luyện trên ImageNet‑21k (14 triệu ảnh) thì hai bên
xấp xỉ nhau, và chỉ với JFT‑300M (303 triệu ảnh) ViT mới vượt. Nhưng “ViT luôn thắng CNN” là một hiểu lầm:
- DeiT (Touvron và cộng sự, 2020) cho ViT học riêng trên ImageNet đạt 81,8% nhờ một công thức tốt: tăng cường dữ liệu
  mạnh (RandAugment, Mixup, CutMix, xóa ngẫu nhiên), AdamW, 300 lượt. Thêm chưng cất tri thức từ một CNN thầy
  (RegNetY‑16GF, 82,9%) qua một token chưng cất, nó lên 83,4% (Bảng 2–3 của DeiT).
- Với cùng lượng tính toán, mạng tích chập hiện đại theo kịp ViT: ConvNeXt (Liu và cộng sự, 2022) trên ImageNet, và
  NFNet tiền huấn luyện trên JFT‑4B (Smith và cộng sự, 2023) ngang các ViT có cùng ngân sách tính toán.

Điều ViT thực sự mang lại là một kiến trúc chung cho ảnh và chữ, dễ mở rộng, chứ không phải một độ chính xác cao hơn
sẵn có. Ngày nay ViT, thường được huấn luyện kiểu CLIP (Bài 22), là bộ mã hóa ảnh của phần lớn các mô hình đa phương thức công bố kiến trúc.

## 12 · Dịch ảnh: ViT gần MLP hơn là gần CNN

Thiên kiến quy nạp thứ hai của tích chập là phép dịch: dịch nét chữ thì bản đồ đặc trưng dịch theo, và max pooling
bỏ qua được những dịch chuyển nhỏ (Bài 11–12). ViT chỉ giữ được điều đó *bên trong* một mảnh, nhờ cùng một $E$ cho mọi
mảnh. Nhưng mỗi mảnh lại gắn với một vectơ vị trí cố định, nên dịch cả ảnh đi vài điểm ảnh thì mô hình thấy những mảnh
khác hẳn ở những vị trí khác hẳn. Script của demo đo trên 10.000 ảnh test, mỗi ảnh dịch 4 px theo cả bốn hướng (không ảnh
huấn luyện nào bị dịch):

In [ ]:
for gid, name in (("lenet", "LeNet"), ("vit4", "ViT có vị trí"), ("mlp", "MLP"), ("vit4-nopos", "ViT không vị trí")):
    g_ = group(gid)
    shift4 = [s["shift"]["acc"][4] for s in g_["seeds"]]
    print(f"{name:<17} ảnh gốc {span([s['test'] for s in g_['seeds']]):>15}  →  dịch 4 px {span(shift4):>15}")

Dịch 4 px, LeNet còn khoảng hai phần ba, ViT có vị trí chưa tới một nửa, gần với MLP: mỗi mảnh gắn với một vị trí cố
định, giống mỗi điểm ảnh gắn với một trọng số riêng trong MLP. ViT không vị trí thì gần như không mất gì. Ô dưới thử
lại trên 500 ảnh test, dịch sang phải 2 px và 4 px, và đếm số dự đoán giữ nguyên như trên ảnh gốc.

In [ ]:
def shift(X, dx, dy):
    """Dời nét dx px sang phải, dy px xuống dưới, phần trống lấp bằng nền (như Bài 11–12)."""
    out = np.zeros_like(X)
    H, W = X.shape[1:3]
    ys, yd = (slice(0, H - dy), slice(dy, H)) if dy >= 0 else (slice(-dy, H), slice(0, H + dy))
    xs, xd = (slice(0, W - dx), slice(dx, W)) if dx >= 0 else (slice(-dx, W), slice(0, W + dx))
    out[:, yd, xd] = X[:, ys, xs]
    return out


N_SH = 300  # ảnh test đầu tiên
for name, p in vit32.items():
    for k in (2, 4):
        pr = predict(p, shift(Xte[:N_SH], k, 0))
        print(
            f"{name:>5}, dịch {k} px: đúng {pct((pr == yte[:N_SH]).mean(), 1)}, "
            f"{pct((pr == pred[name][:N_SH]).mean(), 1)} dự đoán như trên ảnh gốc"
        )
blank = Xte[:N_SH, :, 24:].max((1, 2)) == 0  # 4 cột điểm ảnh bên phải (cột mảnh cuối) chỉ có nền
Xb = Xte[:N_SH][blank]
gap4 = np.abs(forward(vit32["nopos"], shift(Xb, 4, 0)) - forward(vit32["nopos"], Xb)).max()
print(
    f"{int(blank.sum())}/{N_SH} ảnh có cột mảnh cuối trống: dịch 4 px, logit của ViT không vị trí lệch tối đa {sci(gap4)}"
)
assert gap4 < 1e-4

Với ViT không vị trí, dịch 4 px giữ nguyên nhiều dự đoán hơn dịch 2 px. Lý do: 4 px đúng bằng một mảnh. Ảnh dịch 4 px
có đúng các mảnh cũ, chỉ đổi chỗ: cột mảnh bên phải rơi ra ngoài (nó thường trống) và một cột mảnh trống vào từ bên
trái. Túi mảnh không đổi, nên logit không đổi (ô trên: lệch cỡ sai số làm tròn float32). Dịch 2 px thì đường cắt mảnh
rơi vào giữa nét, mọi mảnh có nét đều thành mảnh mới. ViT chỉ bền với những phép dịch đúng bằng bội số của cạnh mảnh;
tích chập trượt từng điểm ảnh nên không có giới hạn đó.

## 13 · Thử thay đổi

Các ô dưới chạy lại `train_vit()` của mục 10 với tham số khác (mỗi lần vài giây). Hãy sửa giá trị rồi chạy lại.

### Mảnh to: còn cần vị trí không?
Với mảnh 14×14, ảnh chỉ còn 4 mảnh, mỗi mảnh là một góc phần tư của chữ số.

In [ ]:
STEPS_BIG = 300  # mảnh to học nhanh, 300 bước là đủ
big = {pos: train_vit(P=14, steps=STEPS_BIG, pos=pos) for pos in (True, False)}
acc_big = {pos: (predict(p, Xte, P=14, heads=2) == yte).mean() for pos, p in big.items()}
print(f"mảnh 14×14, {STEPS_BIG} bước: có vị trí {pct(acc_big[True])}, không vị trí {pct(acc_big[False])}")
print(f"(mảnh 7×7, 400 bước ở mục 10: có vị trí {pct(acc_small[True])}, không vị trí {pct(acc_small[False])})")
assert abs(acc_big[True] - acc_big[False]) < 0.02

Với 4 mảnh to, bỏ vị trí gần như không mất gì. Chữ số MNIST luôn được căn giữa, nên chỉ nhìn nội dung một góc phần tư
(nét cong mở về phía nào, nằm sát cạnh nào) là đã đoán được nó là góc nào. Mảnh càng to, mỗi mảnh càng tự mang thông
tin về vị trí của nó; mảnh càng nhỏ, mô hình càng cần $E_{\text{pos}}$.

Những thứ khác đáng thử:
- `train_vit(seed=1)`, `seed=2`: mỗi seed cho một kết quả hơi khác. So hai cấu hình chỉ khi chênh lệch lớn hơn khoảng
  dao động đó.
- `train_vit(steps=1000)`: lâu gấp 2,5 lần; độ chính xác tăng tiếp.
- `train_vit(P=4)`: 49 mảnh nhỏ, mỗi bước đắt hơn nhiều (mục 6), và trong 400 bước học chậm hơn mảnh 7×7.
- Trong `forward`, thay token [CLS] bằng trung bình mọi mảnh (`hf = layer_norm(h[:, 1:].mean(1), ...)`) và sửa
  `backward` cho khớp: đó là GAP của mục 11.

## 14 · Giới hạn và bước tiếp theo

- **Chi phí tăng theo bình phương số token.** Ảnh độ phân giải cao hay mảnh nhỏ cho rất nhiều token. Các biến thể như
  Swin dùng lại một phần ý của tích chập (chú ý trong cửa sổ cục bộ) để giảm chi phí đó.
- **ViT cần dữ liệu hoặc tăng cường dữ liệu.** Không có thiên kiến của tích chập, nó phải tự học tính cục bộ và tính
  bền với phép dịch; trên MNIST, với cùng công thức, LeNet thắng ở mọi cỡ dữ liệu ta thử.
- **Bản đồ attention dễ đọc sai.** Attention của [CLS] ở lớp cuối dồn vào các token góc chẳng liên quan gì tới vật
  trong ảnh; nó cho biết thông tin đi đường nào, không phải vì sao mô hình đoán như vậy.

Điểm mạnh của ViT là sự đơn giản: ảnh thành một dãy token, rồi dùng đúng Transformer của chữ viết. Nhờ đó, cùng một
kiến trúc học được trên hàng trăm triệu cặp ảnh–chú thích; Bài 22 chạy bộ mã hóa ảnh ViT‑B/32 của CLIP như vậy.

Đến đây, mọi mô hình của khóa học đều đọc ảnh để cho ra một nhãn. Chương tiếp theo đi theo chiều ngược lại: từ vài con
số sinh ra một bức ảnh. Bài 19 bắt đầu bằng autoencoder, mạng nén một chữ số 784 điểm ảnh thành vài số rồi dựng lại nó.

## Tóm tắt
- ViT cắt ảnh thành $N = HW/P^2$ mảnh: chữ số 28×28 với mảnh 4×4 cho 49 mảnh + [CLS] = 50 token; ảnh 224×224 với
  mảnh 16×16 cho 197 token.
- Duỗi mảnh rồi nhân với $E$ đúng bằng một lớp tích chập $D$ bộ lọc $P \times P$, bước trượt $P$ (chênh lệch cỡ
  $10^{-16}$, chỉ là sai số làm tròn).
- Hai ViT của demo (79.690 tham số): có vị trí đúng 97,83%, không vị trí 89,07%. Không vị trí, xáo mảnh không đổi một
  logit nào: mô hình chỉ thấy một túi mảnh.
- ViT MNIST dựa nhiều vào vị trí (xáo mảnh còn 18,76–21,01%) mà không học ra lưới 2‑D; ViT‑Ti/16 trên ImageNet thì có
  (196/196 mảnh giống nhất với một mảnh kề).
- ViT‑Ti có đầu nhìn gần (26 px) và đầu nhìn xa ngay ở khối 1; từ khối 8, 3–4 token ở góc ảnh có chuẩn lớn và nhận
  22–53% attention của [CLS] ở khối 12.
- Cùng công thức huấn luyện, LeNet hơn ViT ở mọi cỡ dữ liệu (8,5 điểm ở 1.000 ảnh, 1,4 điểm ở 55.000 ảnh); dịch ±2 px
  khi học giúp ViT nhiều hơn giúp LeNet.

## Câu hỏi ôn tập
1. Ảnh 384×384 với mảnh 16×16 cho bao nhiêu token? Nếu giảm mảnh xuống 8×8 thì ma trận chú ý lớn lên khoảng bao nhiêu
   lần?
2. Vì sao phép chiếu mảnh $x_p E$ là một lớp tích chập? Bộ lọc, bước trượt và phần đệm của nó là gì?
3. Không có vectơ vị trí, xáo thứ tự các mảnh thì logit của ViT thay đổi thế nào? Vì sao?
4. Mô hình MNIST có vị trí sụp khi xáo mảnh, nhưng vectơ vị trí của nó không giống một lưới 2‑D. Hai điều đó có mâu
   thuẫn không?
5. Vì sao ViT không vị trí giữ nguyên dự đoán khi dịch ảnh 4 px tốt hơn khi dịch 2 px?
6. “ViT tốt hơn CNN” đúng trong điều kiện nào? Kết quả trên MNIST và bài báo DeiT nói gì về câu đó?

<details><summary>Gợi ý đáp án</summary>

1. $(384/16)^2 + 1 = 577$ token. Mảnh 8×8 cho $48^2 + 1 = 2.305$ token, ma trận chú ý $T \times T$ lớn gấp khoảng 16
   lần.
2. Cột $d$ của $E$ là $P^2 C$ trọng số; xếp lại thành $P \times P \times C$ thì là một bộ lọc. Nhân mảnh với cột đó là
   tích chập tại một vị trí. Bước trượt bằng $P$ và không đệm, nên các cửa sổ vừa khít lưới mảnh, không chồng nhau.
3. Không đổi (chỉ lệch sai số làm tròn). Mọi phép tính trên từng token đều giống nhau, còn phép chú ý là tổng có trọng
   số trên tập token, nên đổi chỗ đầu vào chỉ đổi chỗ đầu ra; [CLS] không phải mảnh nên đầu ra của nó giữ nguyên.
4. Không. Mô hình cần biết mảnh nào ở ô nào (mỗi vectơ như một số nhà riêng), nhưng không cần biết ô nào cạnh ô nào,
   vì chữ số MNIST luôn được căn giữa. Trên ảnh thật, vật nằm ở bất cứ đâu, nên ViT‑Ti học được quan hệ lân cận.
5. 4 px đúng bằng cạnh mảnh: ảnh dịch có cùng các mảnh, chỉ đổi chỗ, nên túi mảnh không đổi. Dịch 2 px cắt nét theo
   đường khác, cho các mảnh mới.
6. Khi có rất nhiều dữ liệu tiền huấn luyện (bài báo ViT: JFT‑300M) hoặc một công thức huấn luyện mạnh (DeiT: tăng
   cường dữ liệu, AdamW, 300 lượt, chưng cất). Trên MNIST với cùng công thức, LeNet thắng ở mọi cỡ dữ liệu; với cùng lượng
   tính toán, CNN hiện đại (ConvNeXt, NFNet) theo kịp ViT.
</details>

## Tìm hiểu thêm
- A. Dosovitskiy và cộng sự (2020), [An Image is Worth 16x16 Words](https://arxiv.org/abs/2010.11929): bài báo ViT
  (phương trình 1, thiên kiến quy nạp ở mục 3.1, khoảng cách chú ý ở Hình 7, ablation vị trí ở Bảng 8).
- Mã nguồn và trọng số gốc của Google Research: [vision_transformer](https://github.com/google-research/vision_transformer)
  (phép chiếu mảnh viết bằng `nn.Conv`).
- H. Touvron và cộng sự (2020), [DeiT: Training data-efficient image transformers](https://arxiv.org/abs/2012.12877).
- A. Steiner và cộng sự (2021), [How to train your ViT?](https://arxiv.org/abs/2106.10270): tăng cường dữ liệu, chính quy
  hóa và thêm tính toán thay được khoảng mười lần dữ liệu; nguồn của ViT‑Ti/16 trong demo
  ([timm](https://huggingface.co/timm/vit_tiny_patch16_224.augreg_in21k_ft_in1k)).
- M. Raghu và cộng sự (2021), [Do Vision Transformers See Like Convolutional Neural Networks?](https://arxiv.org/abs/2108.08810).
- J.-B. Cordonnier, A. Loukas & M. Jaggi (2019), [On the Relationship between Self-Attention and Convolutional Layers](https://arxiv.org/abs/1911.03584).
- S. Abnar & W. Zuidema (2020), [Quantifying Attention Flow in Transformers](https://arxiv.org/abs/2005.00928): attention rollout.
- T. Darcet và cộng sự (2023), [Vision Transformers Need Registers](https://arxiv.org/abs/2309.16588).
- Z. Liu và cộng sự (2022), [A ConvNet for the 2020s](https://arxiv.org/abs/2201.03545) (ConvNeXt), và S. Smith và cộng sự
  (2023), [ConvNets Match Vision Transformers at Scale](https://arxiv.org/abs/2310.16764).
- d2l.ai, [Transformer cho thị giác](https://d2l.ai/chapter_attention-mechanisms-and-transformers/vision-transformer.html):
  một ViT viết bằng PyTorch, huấn luyện trên Fashion‑MNIST.

**Bài tiếp theo:** [Bài 19 · Autoencoder & VAE](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/autoencoder-vae/): nén
một ảnh thành vài con số, rồi học cách sinh ảnh mới từ đó.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")